<a href="https://colab.research.google.com/github/Euripedes357/GMSF/blob/main/BRAIN_TUMOR_MRI_Inspetor_da_Rede.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fase 1 — Preparação do Inspetor da Rede

### Passo 1 — Conectar o Drive e conferir o que já existe

Este notebook não treina nada. Ele abre a rede que já foi treinada no notebook original e mostra o que há dentro dela, em 18 gráficos. Por isso o primeiro passo é conectar o Google Drive e conferir se os arquivos de que precisamos estão onde esperamos: o modelo treinado (final.keras), a divisão entre treino e validação (particao_treino_validacao.csv), o histórico do treino (hist_final.csv) e as imagens (archive.zip). Esta célula apenas lê: não escreve nada nas pastas do trabalho original.

A célula também imprime a versão das bibliotecas que vamos usar nos gráficos (Matplotlib, Plotly, Seaborn, SciPy, Graphviz e Pillow). Se alguma não estiver instalada, saberemos agora, antes de montar qualquer gráfico.

In [1]:
# Passo 1 — Conectar o Drive, conferir os arquivos do projeto e as versões das bibliotecas
import os, importlib
from google.colab import drive
drive.mount('/content/drive')

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
ARQUIVOS = {
    'Modelo treinado':           PASTA_PROJETO + '/modelos/final.keras',
    'Partição treino/validação': PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv',
    'Histórico do treino final': PASTA_PROJETO + '/relatorio/hist_final.csv',
    'Imagens (zip)':             PASTA_PROJETO + '/archive.zip',
}
for nome, caminho in ARQUIVOS.items():
    ok = os.path.exists(caminho)
    tam = round(os.path.getsize(caminho) / 1e6, 2) if ok else None
    print(f'{nome:27s} existe? {ok} | tamanho (MB): {tam}')

print()
for lib in ['tensorflow', 'keras', 'numpy', 'pandas', 'matplotlib', 'plotly', 'seaborn', 'scipy', 'graphviz', 'PIL']:
    try:
        m = importlib.import_module(lib)
        print(f'{lib:12s} {getattr(m, "__version__", "(sem versão)")}')
    except ImportError:
        print(f'{lib:12s} NÃO INSTALADA')

Mounted at /content/drive
Modelo treinado             existe? True | tamanho (MB): 4.73
Partição treino/validação   existe? True | tamanho (MB): 0.4
Histórico do treino final   existe? True | tamanho (MB): 0.0
Imagens (zip)               existe? True | tamanho (MB): 164.67

tensorflow   2.21.0
keras        3.13.2
numpy        2.4.6
pandas       2.2.3
matplotlib   3.10.0
plotly       5.24.1
seaborn      0.13.2
scipy        1.16.3
graphviz     0.21
PIL          11.3.0


**Observação:** o Drive foi montado e os quatro arquivos do notebook original foram encontrados: o modelo treinado (final.keras, 4,73 MB), a partição treino/validação (0,4 MB), o histórico do treino final (arquivo pequeno, que aparece como 0,0 MB por arredondamento) e o archive.zip de imagens (164,67 MB, o mesmo tamanho conferido no Passo 1 do original). O modelo ocupa mais que os 388.868 parâmetros sozinhos (cerca de 1,6 MB em precisão simples), provavelmente porque o arquivo guarda também o estado do otimizador Adam, que mantém duas médias por parâmetro. As versões do ambiente coincidem com as do notebook original (TensorFlow 2.21.0, Keras 3.13.2, NumPy 2.4.6, Pandas 2.2.3, Matplotlib 3.10.0, Pillow 11.3.0), o que garante que o modelo será lido do mesmo jeito em que foi gravado. As bibliotecas novas para os gráficos já estão instaladas: Plotly 5.24.1, Seaborn 0.13.2, SciPy 1.16.3 e Graphviz 0.21. Nada precisa ser instalado agora. Uma ressalva: o Graphviz listado é a interface em Python; o programa de desenho em si (dot) só será verificado se de fato formos usá-lo.

### Passo 2 — Descompactar as imagens e conferir a partição treino/validação

As imagens prontas (128×128) do notebook original não foram gravadas, então precisamos refazê-las. Esta célula é a primeira metade disso: copia o archive.zip para o disco local do Colab, descompacta em /content/data (o disco local é apagado ao fim da sessão e não toca no Drive) e abre a partição treino/validação gravada no Passo 9 do original. Ela confere se as contagens por classe batem com as de lá (3.909 imagens de treino e 958 de validação) e se todos os caminhos da partição existem no disco. A conversão para 128×128 em tons de cinza fica para o Passo 3.

In [2]:
# Passo 2 — Descompactar as imagens no disco local e conferir a partição treino/validação
import os, zipfile, shutil, glob
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
DATA_DIR = '/content/data'

if not os.path.exists(DATA_DIR + '/Training'):
    shutil.copy(PASTA_PROJETO + '/archive.zip', '/content/archive.zip')
    zipfile.ZipFile('/content/archive.zip').extractall(DATA_DIR)

TRAIN_DIR = glob.glob(DATA_DIR + '/**/Training', recursive=True)[0]
TEST_DIR  = glob.glob(DATA_DIR + '/**/Testing',  recursive=True)[0]
CLASSES   = sorted(os.listdir(TRAIN_DIR))
print('Training:', TRAIN_DIR)
print('Testing :', TEST_DIR)
print('Classes :', CLASSES)

part = pd.read_csv(PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv')
print('\nPartição:', part.shape, '| colunas:', part.columns.tolist())
tab = pd.crosstab(part['classe'], part['particao']).reindex(CLASSES)
tab.loc['Total'] = tab.sum()
display(tab)

existem = part['caminho'].apply(os.path.exists)
print(f'\nCaminhos da partição que existem no disco: {existem.sum()} de {len(part)}')

Training: /content/data/Training
Testing : /content/data/Testing
Classes : ['glioma', 'meningioma', 'notumor', 'pituitary']

Partição: (4867, 5) | colunas: ['classe', 'arquivo', 'caminho', 'grupo', 'particao']


particao,treino,validacao
classe,,
glioma,1120,276
meningioma,1036,258
notumor,629,148
pituitary,1124,276
Total,3909,958



Caminhos da partição que existem no disco: 4867 de 4867


**Observação:** o archive.zip foi descompactado em /content/data em cerca de 8 segundos, com Training e Testing diretamente nessa pasta, e as quatro classes foram reconhecidas na mesma ordem do notebook original (glioma, meningioma, notumor, pituitary). A partição treino/validação gravada no Passo 9 do original tem 4.867 linhas e cinco colunas (classe, arquivo, caminho, grupo e particao) e reproduz exatamente as contagens de lá: treino com 1.120 glioma, 1.036 meningioma, 629 notumor e 1.124 pituitary (3.909 no total), e validação com 276, 258, 148 e 276 (958 no total). Os 4.867 caminhos da partição existem no disco, o que significa que conseguimos reconstruir exatamente as mesmas imagens de treino e validação do original. O desequilíbrio já conhecido continua visível: o notumor tem 629 imagens de treino, contra mais de mil nas outras classes. O conjunto de teste não foi aberto neste notebook; ele permanece intocado, como no original.

### Passo 3 — Converter as imagens da validação para 128×128 em tons de cinza

Esta célula refaz, apenas para as 958 imagens de validação, a padronização do Passo 10 do notebook original: conversão para tons de cinza, acréscimo de bordas pretas centralizadas até a imagem ficar quadrada (sem distorcer a anatomia) e redução para 128×128 pixels pelo método de Lanczos. Os pixels ficam como inteiros de 0 a 255; a divisão por 255 acontece dentro da própria rede, na primeira camada. Os rótulos seguem a ordem glioma=0, meningioma=1, notumor=2 e pituitary=3. A conferência é o formato do resultado (958, 128, 128, 1), pixels de 0 a 255 e a contagem por classe, que deve ser igual à da validação do original.

In [3]:
# Passo 3 — Padronizar as imagens da validação: tons de cinza + bordas pretas até ficar quadrada + 128x128
import os, glob
import numpy as np
import pandas as pd
from PIL import Image

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
TRAIN_DIR = glob.glob('/content/data/**/Training', recursive=True)[0]
CLASSES = sorted(os.listdir(TRAIN_DIR))
part = pd.read_csv(PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv')

TAM = 128
def preparar(caminho):
    with Image.open(caminho) as img:
        img = img.convert('L')                                  # tons de cinza
        w, h = img.size
        lado = max(w, h)
        quadrada = Image.new('L', (lado, lado), 0)              # fundo preto
        quadrada.paste(img, ((lado - w) // 2, (lado - h) // 2)) # imagem centralizada
        return np.asarray(quadrada.resize((TAM, TAM), Image.LANCZOS), dtype=np.uint8)

mapa = {c: i for i, c in enumerate(CLASSES)}                    # glioma=0, meningioma=1, notumor=2, pituitary=3
val = part[part['particao'] == 'validacao'].reset_index(drop=True)
X_val = np.stack([preparar(c) for c in val['caminho']])[..., np.newaxis]
y_val = val['classe'].map(mapa).values.astype(np.int64)

print(f'X_val {X_val.shape} {X_val.dtype} | pixels de {X_val.min()} a {X_val.max()}')
print('Imagens por classe:', dict(zip(CLASSES, np.bincount(y_val, minlength=4))))

X_val (958, 128, 128, 1) uint8 | pixels de 0 a 255
Imagens por classe: {'glioma': np.int64(276), 'meningioma': np.int64(258), 'notumor': np.int64(148), 'pituitary': np.int64(276)}


**Observação:** as 958 imagens de validação foram padronizadas em cerca de 3 segundos e resultaram num conjunto de formato (958, 128, 128, 1), do tipo inteiro de 8 bits, com pixels de 0 a 255, como previsto no Passo 10 do notebook original. A contagem por classe confere para glioma (276) e meningioma (258); as outras duas classes somam as 424 imagens restantes, compatível com os 148 notumor e 276 pituitary da partição do Passo 2. O resultado ainda não passou pela divisão por 255, que acontece dentro da rede, na primeira camada. A ordem das imagens segue a da partição gravada, e os rótulos seguem glioma=0, meningioma=1, notumor=2 e pituitary=3. Como a conversão é determinística (sem sorteios), refazê-la gera exatamente as mesmas imagens que o notebook original usou na validação. A prova definitiva dessa equivalência virá no Passo 4: se o modelo treinado, aplicado a estas imagens, reproduzir o resultado do original, o preparo está correto.

### Passo 4 — Carregar o modelo treinado e conferir se ele reproduz o resultado do original

Antes de desenhar qualquer gráfico, precisamos ter certeza de que o modelo carregado e as imagens reconstruídas são os mesmos do notebook original. O teste é simples: o final.keras (a rede treinada, no ponto da época 28) é carregado, avaliado nas 958 imagens de validação, e o resultado é comparado com o do Passo 27 do original: perda 0,328, acurácia de 91,13% (873 acertos em 958) e revocação por classe de 90,9% para glioma, 86,8% para meningioma, 90,5% para notumor e 95,7% para pituitary. A célula também lista as camadas do modelo, com forma de saída e parâmetros. Se os números baterem, a reconstrução está validada; se não, investigamos antes de seguir.

In [4]:
# Passo 4 — Carregar o final.keras e conferir o resultado na validação (deve reproduzir o Passo 27 do original)
import numpy as np
import keras

modelo = keras.models.load_model(PASTA_PROJETO + '/modelos/final.keras')

print('Camadas do modelo:')
for camada in modelo.layers:
    try:
        forma = tuple(camada.output.shape[1:])
    except Exception:
        forma = '—'
    print(f'  {camada.name:18s} {camada.__class__.__name__:24s} {str(forma):18s} {camada.count_params():>8,}')
print(f'Total de parâmetros: {modelo.count_params():,} (esperado: 388,868)\n')

x = X_val.astype('float32')
perda, acc = modelo.evaluate(x, y_val, verbose=0)
prob = modelo.predict(x, verbose=0)
pred = prob.argmax(axis=1)

print(f'Perda na validação    : {perda:.3f}  (original: 0.328)')
print(f'Acurácia na validação : {100 * acc:.2f}%  (original: 91.13%)')
print(f'Acertos               : {(pred == y_val).sum()} de {len(y_val)}  (original: 873 de 958)\n')

print('Revocação por classe (original: glioma 90.9 | meningioma 86.8 | notumor 90.5 | pituitary 95.7):')
for i, c in enumerate(CLASSES):
    print(f'  {c:11s} {100 * (pred[y_val == i] == i).mean():.1f}%')

Camadas do modelo:
  imagem             InputLayer               (128, 128, 1)             0
  rescaling_37       Rescaling                (128, 128, 1)             0
  conv2d_116         Conv2D                   (128, 128, 32)          320
  re_lu_116          ReLU                     (128, 128, 32)            0
  max_pooling2d_116  MaxPooling2D             (64, 64, 32)              0
  conv2d_117         Conv2D                   (64, 64, 64)         18,496
  re_lu_117          ReLU                     (64, 64, 64)              0
  max_pooling2d_117  MaxPooling2D             (32, 32, 64)              0
  conv2d_118         Conv2D                   (32, 32, 128)        73,856
  re_lu_118          ReLU                     (32, 32, 128)             0
  max_pooling2d_118  MaxPooling2D             (16, 16, 128)             0
  conv2d_119         Conv2D                   (16, 16, 256)       295,168
  re_lu_119          ReLU                     (16, 16, 256)             0
  max_pooling2d_119

**Observação:** o final.keras foi carregado e avaliado nas 958 imagens de validação reconstruídas, e o resultado coincide com o do Passo 27 do notebook original em todos os números conferidos: 388.868 parâmetros, perda de 0,328, acurácia de 91,13% (873 acertos em 958) e revocação de 90,9% para glioma, 86,8% para meningioma, 90,5% para notumor e 95,7% para pituitary. Essa coincidência prova duas coisas ao mesmo tempo: o modelo gravado é o ponto da época 28 do original, e o preparo das imagens refeito neste notebook é equivalente ao de lá. A partir daqui, todo gráfico que sair deste notebook descreve a rede real do projeto. Os nomes das camadas gravadas no arquivo carregam sufixos automáticos do Keras (por exemplo conv2d_118, re_lu_119 e dense_20), porque o modelo final foi construído por uma função que não as nomeia; por isso as camadas serão sempre localizadas pelo tipo e pela ordem, nunca pelo nome. A rede tem quatro camadas convolucionais (com 32, 64, 128 e 256 filtros), uma camada densa de 4 saídas, e as camadas ReLU, MaxPooling, média global e Dropout não têm parâmetros. O Dropout (taxa 0,3) só atua durante o treino e fica desligado quando a rede é aplicada a imagens, como aqui.

## Fase 2 — Inventário de parâmetros

### Passo 5 — Inventário das camadas: o que a rede tem dentro

Esta célula abre o modelo treinado e monta uma tabela com as camadas que guardam parâmetros: as quatro convolucionais e a camada densa de saída. Para cada uma ela mostra o formato do tensor de pesos (altura, largura, canais de entrada e número de filtros, no caso da convolução), quantos parâmetros são pesos e quantos são biases (viéses), o total da camada e a fatia que ela representa dos 388.868 parâmetros da rede. As demais camadas (ReLU, MaxPooling, média global e Dropout) não têm parâmetros e não entram na tabela. Este inventário é a base dos gráficos de parâmetros, pesos e biases que vêm a seguir.

In [6]:
# Passo 5 — Inventário das camadas com parâmetros: formato do tensor, pesos, biases e fatia do total
import pandas as pd

total = modelo.count_params()
linhas, n_conv = [], 0
for camada in modelo.layers:
    if not camada.weights:
        continue
    pesos, vieses = camada.get_weights()
    tipo = camada.__class__.__name__
    if tipo == 'Conv2D':
        n_conv += 1
        rotulo = f'Conv {n_conv} ({camada.filters} filtros)'
    else:
        rotulo = 'Dense (4 saídas)'
    linhas.append({
        'Camada': rotulo,
        'Tipo': tipo,
        'Formato dos pesos': str(pesos.shape),
        'Pesos': pesos.size,
        'Biases': vieses.size,
        'Total': pesos.size + vieses.size,
        '% do total': round(100 * (pesos.size + vieses.size) / total, 1),
    })

inv = pd.DataFrame(linhas)
inv.loc[len(inv)] = ['Total', '', '', inv['Pesos'].sum(), inv['Biases'].sum(), inv['Total'].sum(), 100.0]
display(inv.set_index('Camada'))
print(f'\nConferência: soma dos parâmetros = {inv["Total"].iloc[:-1].sum():,} | modelo.count_params() = {total:,}')

,Tipo,Formato dos pesos,Pesos,Biases,Total,% do total
Camada,,,,,,
Conv 1 (32 filtros),Conv2D,"(3, 3, 1, 32)",288,32,320,0.1
Conv 2 (64 filtros),Conv2D,"(3, 3, 32, 64)",18432,64,18496,4.8
Conv 3 (128 filtros),Conv2D,"(3, 3, 64, 128)",73728,128,73856,19.0
Conv 4 (256 filtros),Conv2D,"(3, 3, 128, 256)",294912,256,295168,75.9
Dense (4 saídas),Dense,"(256, 4)",1024,4,1028,0.3
Total,,,388384,484,388868,100.0



Conferência: soma dos parâmetros = 388,868 | modelo.count_params() = 388,868


**Observação:** o inventário confirma, camada por camada, a arquitetura do notebook original. A primeira convolução tem 320 parâmetros (3×3×1×32 = 288 pesos, mais 32 biases, um por filtro); a segunda, 18.496; a terceira, 73.856; a quarta, 295.168; e a camada densa de saída, 1.028 (256×4 = 1.024 pesos, mais 4 biases). A soma fecha em 388.868, igual ao total informado pelo modelo, dos quais 388.384 são pesos e apenas 484 são biases (0,12% do total). A distribuição é muito desigual: a última convolução concentra 75,9% de todos os parâmetros, a terceira 19,0%, a segunda 4,8%, e a primeira e a camada densa juntas ficam abaixo de 0,5%. A razão está na fórmula do número de pesos de uma convolução, 3 × 3 × (canais de entrada) × (filtros): ela cresce com o produto entre os canais que entram e os filtros que saem, e esse produto aumenta a cada bloco (1×32, 32×64, 64×128 e 128×256). O tamanho da imagem não entra na conta, porque o mesmo filtro é reaproveitado em todas as posições; por isso a primeira convolução, que percorre 128×128 posições, tem só 320 parâmetros. Este quadro é a base dos gráficos de parâmetros, pesos e biases que vêm a seguir.

### Passo 6 — Gráfico 07: quantidade de parâmetros por camada

O Gráfico 07 do catálogo mostra quantos parâmetros treináveis cada camada possui, separando pesos e biases. Aqui ele é construído com os números reais do modelo final, extraídos do inventário do Passo 5. Como os biases são uma fração mínima do total (484 de 388.868), o gráfico tem dois painéis: à esquerda, o total de parâmetros de cada camada em escala linear, com o valor e a porcentagem escritos sobre cada barra, para mostrar onde a rede concentra sua capacidade; à direita, pesos e biases lado a lado em escala logarítmica, que permite enxergar os dois grupos mesmo com diferença de ordens de grandeza. O gráfico é interativo: passar o mouse sobre uma barra mostra o valor exato.

In [8]:
# Passo 6 — Gráfico 07: parâmetros por camada (dados reais do modelo final)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

d = inv.iloc[:-1]                                    # sem a linha Total
rot = [r.split(' (')[0] for r in d['Camada']]        # Conv 1, Conv 2, Conv 3, Conv 4, Dense
pt = lambda v: f'{int(v):,}'.replace(',', '.')       # 295168 -> 295.168

fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.12,
                    subplot_titles=('Total por camada (escala linear)', 'Pesos e biases (escala logarítmica)'))

fig.add_trace(go.Bar(x=rot, y=d['Total'], marker_color='#1F4E79', showlegend=False,
                     text=[f'{pt(t)}<br>{str(p).replace(".", ",")}%' for t, p in zip(d['Total'], d['% do total'])],
                     textposition='outside', hovertemplate='%{x}: %{y:,} parâmetros<extra></extra>'), row=1, col=1)
fig.add_trace(go.Bar(x=rot, y=d['Pesos'], name='Pesos', marker_color='#1F4E79',
                     hovertemplate='%{x}: %{y:,} pesos<extra></extra>'), row=1, col=2)
fig.add_trace(go.Bar(x=rot, y=d['Biases'], name='Biases', marker_color='#C9A227',
                     hovertemplate='%{x}: %{y:,} biases<extra></extra>'), row=1, col=2)

fig.update_yaxes(range=[0, d['Total'].max() * 1.2], title_text='Parâmetros', row=1, col=1)
fig.update_yaxes(type='log', title_text='Parâmetros (log)', row=1, col=2)
fig.update_layout(title='Gráfico 07 — Quantidade de parâmetros por camada (modelo final do projeto)',
                   barmode='group', template='plotly_white', height=520,
                  legend=dict(orientation='h', y=-0.15, x=0.62))
fig.show()

**Observação:** o Gráfico 07, construído com os números reais do modelo final, mostra que a capacidade da rede está concentrada no fim da parte convolucional: a Conv 4 tem 295.168 parâmetros (75,9% do total), a Conv 3 tem 73.856 (19,0%), a Conv 2 tem 18.496 (4,8%), a Conv 1 tem 320 (0,1%) e a camada densa de saída tem 1.028 (0,3%). Nenhuma das camadas do início tem parâmetros em quantidade comparável à última convolução, apesar de trabalharem sobre imagens muito maiores (128×128 contra 16×16): o que pesa é a quantidade de canais e de filtros, e não o tamanho do mapa. No painel da direita, em escala logarítmica, os pesos aparecem sempre acima dos biases, e a distância entre as barras aumenta de camada em camada. Isso tem uma explicação simples: numa convolução 3×3, a razão entre pesos e biases é 9 vezes o número de canais de entrada (9 na Conv 1, 288 na Conv 2, 576 na Conv 3 e 1.152 na Conv 4), e na camada densa é 256. Em conjunto, os biases somam 484 parâmetros, o que explica por que seriam invisíveis em escala linear. A leitura é descritiva: a quantidade de parâmetros mede a capacidade de cada camada, mas não diz quanto cada uma contribui para a decisão da rede; isso será examinado nos gráficos de pesos e de ativações.

### Passo 7 — Gráfico 08: mapa de calor dos pesos

O Gráfico 08 do catálogo usa cores para mostrar o sinal e a intensidade dos pesos. Aqui ele é feito com os pesos reais do modelo final, em dois painéis. No painel de cima estão os 32 filtros 3×3 da primeira convolução: cada coluna é um filtro e cada linha é uma posição dentro do filtro, de (0,0) no canto superior esquerdo a (2,2) no canto inferior direito. No painel de baixo está a matriz de pesos da camada densa final: cada linha é uma das quatro classes e cada coluna é uma das 256 características que chegam da média global. Azul indica peso negativo, vermelho indica peso positivo, e o branco é próximo de zero. Cada painel tem sua própria escala, simétrica em torno do zero, porque os pesos das duas camadas têm magnitudes diferentes; por isso cores iguais em painéis diferentes não representam o mesmo valor. A leitura é descritiva: o sinal e o tamanho de um peso não bastam para dizer quanto uma característica importa para a decisão.

In [9]:
# Passo 7 — Gráfico 08: mapa de calor dos pesos reais (Conv 1 e camada densa)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conv1 = [l for l in modelo.layers if l.__class__.__name__ == 'Conv2D'][0]
densa = [l for l in modelo.layers if l.__class__.__name__ == 'Dense'][0]

W1 = conv1.get_weights()[0]                        # (3, 3, 1, 32)
M1 = W1[:, :, 0, :].reshape(9, 32)                 # linhas: posição (i,j); colunas: filtro
Wd = densa.get_weights()[0].T                      # (4 classes, 256 características)

pos = [f'({i},{j})' for i in range(3) for j in range(3)]
lim1, limd = np.abs(M1).max(), np.abs(Wd).max()

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.2, row_heights=[0.55, 0.45],
                    subplot_titles=('Conv 1: 32 filtros 3×3 (colunas = filtros, linhas = posição no filtro)',
                                    'Camada densa: pesos por classe e por característica (256)'))

fig.add_trace(go.Heatmap(z=M1, x=list(range(1, 33)), y=pos, colorscale='RdBu', reversescale=True,
                         zmin=-lim1, zmax=lim1, xgap=1, ygap=1,
                         colorbar=dict(title='Peso', y=0.8, len=0.4, x=1.02),
                         hovertemplate='filtro %{x}, posição %{y}: %{z:.3f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Heatmap(z=Wd, x=list(range(1, 257)), y=CLASSES, colorscale='RdBu', reversescale=True,
                         zmin=-limd, zmax=limd,
                         colorbar=dict(title='Peso', y=0.2, len=0.35, x=1.02),
                         hovertemplate='característica %{x}, classe %{y}: %{z:.3f}<extra></extra>'), row=2, col=1)

fig.update_yaxes(autorange='reversed', row=1, col=1)
fig.update_xaxes(title_text='Filtro', row=1, col=1)
fig.update_xaxes(title_text='Característica (saída da média global)', row=2, col=1)
fig.update_layout(title='Gráfico 08 — Mapa de calor dos pesos (modelo final do projeto)',
                  template='plotly_white', height=720)
fig.show()

print(f'Conv 1: pesos de {M1.min():.3f} a {M1.max():.3f} | Camada densa: pesos de {Wd.min():.3f} a {Wd.max():.3f}')

Conv 1: pesos de -1.341 a 1.112 | Camada densa: pesos de -0.578 a 0.571


**Observação:** o painel de cima mostra os 288 pesos da primeira convolução como 32 filtros 3×3 (uma coluna por filtro e uma linha por posição). Os filtros são claramente diferentes entre si e misturam pesos positivos e negativos dentro do mesmo filtro, sem um padrão único; os valores são da ordem de ±1, o que é relativamente grande, porque cada neurônio dessa camada soma apenas 9 entradas. O painel de baixo mostra a camada densa final, com 4 linhas (uma por classe) e 256 colunas (uma por característica que chega da média global). Aqui os valores são menores, da ordem de ±0,5, e a imagem tem o aspecto de listras verticais finas, com sinais misturados em cada linha: nenhuma característica isolada domina visualmente, e cada classe se apoia em muitas características ao mesmo tempo, em vez de depender de poucas. Como as características que chegam à camada densa vêm de uma ReLU seguida de média e, portanto, nunca são negativas, um peso positivo empurra a pontuação da classe para cima quando aquela característica está ativa, e um peso negativo empurra para baixo. Como o softmax só compara as classes entre si, para uma mesma característica o que conta é a diferença entre os pesos das quatro classes, e não o valor absoluto. A leitura é descritiva: o sinal e o tamanho de um peso, vistos isoladamente, não dizem quanto uma característica importa para a decisão, e o mapa de calor dos filtros da primeira camada não revela sozinho o que cada filtro detecta; isso aparecerá nos mapas de ativação.

### Passo 8 — Gráfico 10: distribuição dos pesos por camada (violino e boxplot)

O Gráfico 10 do catálogo resume os pesos de cada camada pela sua distribuição: o violino mostra a forma (onde os pesos se concentram e como se espalham) e o boxplot interno marca a mediana e os quartis. Aqui ele é feito com os pesos reais do modelo final, para as quatro convoluções e a camada densa, todos na mesma escala vertical para que as camadas possam ser comparadas. Como a última convolução tem quase 295 mil pesos, o desenho do violino usa uma amostra aleatória de até 20.000 pesos por camada (semente 42; nas camadas menores, todos os pesos), enquanto a tabela impressa abaixo do gráfico traz as estatísticas exatas (mínimo, quartis, mediana, máximo, média e desvio-padrão) calculadas com todos os pesos. Os biases não entram aqui: eles têm gráfico próprio.

In [10]:
# Passo 8 — Gráfico 10: distribuição dos pesos por camada (violino + boxplot), pesos reais do modelo final
import numpy as np
import pandas as pd
import plotly.graph_objects as go

rng = np.random.default_rng(42)
camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

fig = go.Figure()
linhas = []
for r, camada in zip(rot, camadas):
    w = camada.get_weights()[0].ravel()                                   # todos os pesos da camada
    amostra = rng.choice(w, size=min(20000, w.size), replace=False)       # só para desenhar
    fig.add_trace(go.Violin(y=amostra, name=r, box_visible=True, points=False, showlegend=False,
                            line_color='#1F4E79', fillcolor='rgba(31,78,121,0.35)',
                            meanline=dict(visible=True, color='#C9A227', width=2)))
    q1, med, q3 = np.percentile(w, [25, 50, 75])
    linhas.append({'Camada': r, 'Pesos': w.size, 'Mínimo': w.min(), 'Q1': q1, 'Mediana': med,
                   'Q3': q3, 'Máximo': w.max(), 'Média': w.mean(), 'Desvio-padrão': w.std()})

fig.update_layout(title='Gráfico 10 — Distribuição dos pesos por camada (modelo final do projeto)',
                  yaxis_title='Peso', template='plotly_white', height=520)
fig.show()

tab = pd.DataFrame(linhas).set_index('Camada')
display(tab.round(4))

,Pesos,Mínimo,Q1,Mediana,Q3,Máximo,Média,Desvio-padrão
Camada,,,,,,,,
Conv 1,288,-1.3414,-0.3864,0.0317,0.3741,1.1120,0.0118,0.5223
Conv 2,18432,-0.8529,-0.0930,-0.0055,0.0824,0.8012,-0.0074,0.1419
Conv 3,73728,-0.8520,-0.0873,-0.0197,0.0431,0.7388,-0.0264,0.1099
Conv 4,294912,-0.6584,-0.0635,-0.0121,0.0369,0.7476,-0.0152,0.0874
Dense,1024,-0.5784,-0.1327,0.0118,0.1142,0.5709,-0.0075,0.1604


Quanto à forma, o gráfico mostra que a Conv 1 tem um corpo largo, com a caixa central indo de cerca de −0,4 a +0,35, o que indica pesos bem espalhados. Nas Conv 2, 3 e 4 o corpo é estreito e concentrado em torno de zero, com caudas longas e finas, e a camada densa fica em um meio-termo, com corpo um pouco mais largo. Todos os violinos são aproximadamente simétricos em relação ao zero, sem tendência clara para pesos positivos ou negativos. As pontas dos violinos passam um pouco dos extremos reais, porque o gráfico suaviza a curva; os valores exatos são os da tabela.

### Passo 9 — Gráfico 12: distribuição dos biases por camada

O Gráfico 12 do catálogo mostra, para cada camada, como os biases (os termos aditivos de cada filtro ou neurônio) se distribuem. Aqui ele é feito com os biases reais do modelo final: 32 na Conv 1, 64 na Conv 2, 128 na Conv 3, 256 na Conv 4 e 4 na camada densa, 484 no total. Como são poucos, cada bias é desenhado como um ponto sobre o boxplot da sua camada, e a linha pontilhada marca o zero. Um dado importante para a interpretação: no Keras, os biases das camadas começam em zero, de modo que qualquer valor diferente de zero foi aprendido durante o treino. A tabela impressa abaixo do gráfico traz as estatísticas exatas e a contagem de biases positivos e negativos por camada.

In [11]:
# Passo 9 — Gráfico 12: distribuição dos biases por camada (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

fig = go.Figure()
linhas = []
for r, camada in zip(rot, camadas):
    b = camada.get_weights()[1]                                   # biases da camada
    fig.add_trace(go.Box(y=b, name=r, boxpoints='all', jitter=0.5, pointpos=0, showlegend=False,
                         marker=dict(color='#C9A227', size=5, opacity=0.8),
                         line=dict(color='#1F4E79'), fillcolor='rgba(31,78,121,0.15)'))
    q1, med, q3 = np.percentile(b, [25, 50, 75])
    linhas.append({'Camada': r, 'Biases': b.size, 'Mínimo': b.min(), 'Q1': q1, 'Mediana': med, 'Q3': q3,
                   'Máximo': b.max(), 'Média': b.mean(), 'Desvio-padrão': b.std(),
                   'Positivos': int((b > 0).sum()), 'Negativos': int((b < 0).sum())})

fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(title='Gráfico 12 — Distribuição dos biases por camada (modelo final do projeto)',
                  yaxis_title='Bias', template='plotly_white', height=520)
fig.show()

tab = pd.DataFrame(linhas).set_index('Camada')
display(tab.round(4))

,Biases,Mínimo,Q1,Mediana,Q3,Máximo,Média,Desvio-padrão,Positivos,Negativos
Camada,,,,,,,,,,
Conv 1,32,-0.0548,-0.0154,-0.0071,0.0035,0.0689,-0.0022,0.0276,12,20
Conv 2,64,-0.2192,-0.0468,-0.0061,0.0290,0.1165,-0.0091,0.0622,24,40
Conv 3,128,-0.2551,-0.0660,-0.0222,0.0227,0.1018,-0.0286,0.0643,45,83
Conv 4,256,-0.1999,-0.0554,0.0007,0.0297,0.1501,-0.0127,0.0648,131,125
Dense,4,-0.0947,-0.0941,-0.0143,0.0815,0.1300,0.0017,0.0987,2,2


**Observação:** a tabela confirma que os biases são pequenos: os mínimos e máximos ficam entre −0,2551 (Conv 3) e +0,1501 (Conv 4). Como os biases começam em zero no Keras, tudo isso foi aprendido no treino. A Conv 1 é a mais concentrada (32 biases, de −0,0548 a +0,0689, desvio-padrão de 0,0276). A partir da Conv 2 a dispersão sobe e depois se estabiliza: o desvio-padrão é 0,0622 na Conv 2, 0,0643 na Conv 3 e 0,0648 na Conv 4, ou seja, não cresce junto com o número de filtros (64, 128, 256). A camada densa tem só 4 biases, um por classe, com desvio de 0,0987; dois deles estão quase iguais ao mínimo (−0,0947 e −0,0941), e o quartil Q1 coincide com o mínimo por esse motivo. Comparando com os pesos do Passo 8, na Conv 1 os pesos (desvio 0,522) são cerca de 19 vezes mais dispersos que os biases, mas essa distância diminui nas camadas profundas: na Conv 4 os desvios são 0,087 (pesos) e 0,065 (biases), de ordem parecida. As médias das camadas convolucionais são levemente negativas (de −0,0022 a −0,0286), com a Conv 3 mais deslocada, e as medianas também ficam abaixo de zero, exceto na Conv 4 (+0,0007). Um bias negativo exige uma soma maior para o filtro "ligar", então isso sugere uma tendência leve de ativação mais exigente nessas camadas, mas é só uma leitura descritiva: os números sozinhos não provam esse efeito, e a tabela abaixo não diz quais biases importam para o resultado.

### Passo 10 — Gráfico 09: distribuição 3D dos pesos por camada

O Gráfico 09 do catálogo coloca as distribuições de pesos das camadas lado a lado em três dimensões. Aqui ele é feito com os pesos reais do modelo final. O eixo horizontal é o valor do peso, o eixo de profundidade são as cinco camadas com pesos (Conv 1 a Conv 4 e Dense), e a altura é a densidade, isto é, quão concentrados estão os pesos em cada faixa de valores. A densidade é normalizada para que cada camada tenha área total igual a 1, o que permite comparar camadas com números muito diferentes de pesos (288 na Conv 1 contra 294.912 na Conv 4). Cada camada aparece como uma faixa separada, sem unir uma à outra: o espaço entre as faixas fica vazio de propósito, porque não existem dados entre duas camadas. O gráfico é interativo: arraste com o mouse para girar, use a roda para dar zoom e passe o cursor sobre a superfície para ler o valor do peso e a densidade. Abaixo dele, uma pequena tabela traz a densidade no pico de cada camada e a porcentagem de pesos entre −0,1 e +0,1.

In [12]:
# Passo 10 — Gráfico 09: distribuição 3D dos pesos por camada (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

limites = np.linspace(-1.4, 1.2, 66)                 # 65 faixas de largura 0,04
centros = (limites[:-1] + limites[1:]) / 2

y_pos, z_linhas, resumo = [], [], []
for i, (r, camada) in enumerate(zip(rot, camadas)):
    w = camada.get_weights()[0].ravel()               # todos os pesos da camada
    dens, _ = np.histogram(w, bins=limites, density=True)
    y_pos += [i - 0.3, i + 0.3, i + 0.5]              # faixa da camada + linha vazia de separação
    z_linhas += [dens, dens, np.full_like(dens, np.nan)]
    resumo.append({'Camada': r, 'Pesos': w.size, 'Densidade no pico': dens.max(),
                   '% dos pesos entre -0,1 e +0,1': 100 * np.mean(np.abs(w) < 0.1)})

fig = go.Figure(go.Surface(x=centros, y=y_pos[:-1], z=np.array(z_linhas[:-1]),
                           colorscale=[[0, '#F3F6FA'], [0.5, '#1F4E79'], [1, '#C9A227']],
                           colorbar=dict(title='Densidade'),
                           hovertemplate='Peso: %{x:.2f}<br>Densidade: %{z:.2f}<extra></extra>'))
fig.update_layout(title='Gráfico 09 — Distribuição 3D dos pesos por camada (modelo final do projeto)',
                  template='plotly_white', height=650,
                  scene=dict(xaxis_title='Valor do peso', yaxis_title='Camada', zaxis_title='Densidade',
                             yaxis=dict(tickmode='array', tickvals=list(range(5)), ticktext=rot),
                             camera=dict(eye=dict(x=1.6, y=-1.8, z=0.9))))
fig.show()

tab = pd.DataFrame(resumo).set_index('Camada')
display(tab.round(2))

,Pesos,Densidade no pico,"% dos pesos entre -0,1 e +0,1"
Camada,,,
Conv 1,288,1.48,10.42
Conv 2,18432,3.14,55.57
Conv 3,73728,4.14,69.08
Conv 4,294912,5.30,79.21
Dense,1024,2.73,39.94


**Observação:** o gráfico mostra que as distribuições de pesos se concentram cada vez mais perto de zero ao longo das camadas convolucionais. A densidade no pico sobe de 1,48 na Conv 1 para 3,14 na Conv 2 e 4,14 na Conv 3, e na Conv 4 o pico é ainda mais alto, o maior do gráfico. A camada densa fica abaixo das camadas profundas, o que é coerente com o desvio-padrão maior dos seus pesos (0,160, contra 0,087 na Conv 4). A tabela confirma o mesmo movimento: a porcentagem de pesos entre −0,1 e +0,1 passa de 10,42% na Conv 1 para 55,57% na Conv 2 e 69,08% na Conv 3, ou seja, na Conv 3 mais de dois terços dos 73.728 pesos são muito pequenos. Isso combina com os desvios-padrão do Passo 8 (0,522, 0,142, 0,110, 0,087). Os picos das Conv 2 e 3 são mais altos do que seriam em uma distribuição normal com o mesmo desvio-padrão (cerca de 2,8 e 3,6, pela fórmula teórica), o que indica um centro mais afunilado e caudas longas, como já aparecia nos violinos. A Conv 1 é a mais baixa e mais espalhada, mas também a mais irregular, porque tem apenas 288 pesos distribuídos em faixas de 0,04 de largura, e o valor do pico dela deve ser lido com cautela. Cada camada é uma faixa separada: o espaço vazio entre elas não é dado, é só separação visual. A leitura é descritiva e não prova, por si só, que pesos concentrados perto de zero sejam bons ou ruins para a classificação.

### Passo 11 — Gráfico 11: os filtros convolucionais em 3D

O Gráfico 11 do catálogo mostra os filtros de uma camada convolucional em três dimensões. Aqui ele é feito com os filtros reais da primeira camada do modelo final. A Conv 1 tem 32 filtros, e cada um é uma pequena grade de 3×3 pesos (a imagem de entrada tem um só canal, por isso cada filtro tem profundidade 1). No gráfico, cada filtro vira nove barras sobre uma grade de 3 linhas por 3 colunas: a altura da barra é o valor do peso, e a cor segue a mesma convenção do Gráfico 08, vermelho para pesos positivos e azul para negativos, com a mesma escala de cores para os 32 filtros, de modo que as cores sejam comparáveis entre eles. A barra deslizante troca de filtro, e o botão "Percorrer os 32 filtros" faz isso automaticamente, um filtro por vez. Você também pode girar o gráfico com o mouse e dar zoom com a roda. Abaixo do gráfico, o código imprime os nove pesos numéricos do filtro 1, para você conferir com as barras iniciais, e uma tabela com os 8 filtros de maior peso em valor absoluto.

In [13]:
# Passo 11 — Gráfico 11: os 32 filtros 3x3 da primeira camada convolucional (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

W1 = [l for l in modelo.layers if l.weights][0].get_weights()[0]     # forma (3, 3, 1, 32)
nf = W1.shape[3]
K = [W1[:, :, 0, f] for f in range(nf)]                               # 32 filtros 3x3
vmax = float(np.ceil(np.abs(W1).max() * 10) / 10)                     # escala comum a todos os filtros

# geometria fixa: uma barra (caixa) por posição do filtro 3x3
X, Y, I, J, Kk = [], [], [], [], []
for b, (r, c) in enumerate([(r, c) for r in range(3) for c in range(3)]):
    x0, x1, y0, y1 = c - 0.4, c + 0.4, r - 0.4, r + 0.4
    X += [x0, x1, x1, x0] * 2
    Y += [y0, y0, y1, y1] * 2
    o = 8 * b
    for (a, bb, cc) in [(0,1,2),(0,2,3),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]:
        I.append(o + a); J.append(o + bb); Kk.append(o + cc)

def alturas(M):
    z, cor = [], []
    for r in range(3):
        for c in range(3):
            h = float(M[r, c])
            z += [0, 0, 0, 0, h, h, h, h]
            cor += [h] * 8
    return z, cor

z0, c0 = alturas(K[0])
malha = go.Mesh3d(x=X, y=Y, z=z0, i=I, j=J, k=Kk, intensity=c0,
                  colorscale='RdBu', reversescale=True, cmin=-vmax, cmax=vmax,
                  colorbar=dict(title='Peso'), flatshading=True,
                  hovertemplate='Peso: %{intensity:.3f}<extra></extra>')

quadros = []
for f in range(nf):
    z, cor = alturas(K[f])
    quadros.append(go.Frame(name=str(f + 1), data=[go.Mesh3d(z=z, intensity=cor)]))

fig = go.Figure(data=[malha], frames=quadros)
passos = [dict(method='animate', label=str(f + 1),
               args=[[str(f + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
          for f in range(nf)]
fig.update_layout(
    title='Gráfico 11 — Os 32 filtros 3×3 da primeira camada convolucional (modelo final do projeto)',
    template='plotly_white', height=700,
    scene=dict(xaxis=dict(title='Coluna', tickvals=[0, 1, 2], ticktext=['1', '2', '3']),
               yaxis=dict(title='Linha', tickvals=[0, 1, 2], ticktext=['1', '2', '3'], autorange='reversed'),
               zaxis=dict(title='Peso', range=[-vmax, vmax]),
               camera=dict(eye=dict(x=1.7, y=-1.7, z=1.2)), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.8)),
    sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Filtro '), pad=dict(t=90))],
    updatemenus=[dict(type='buttons', direction='left', x=0.0, y=0.0, xanchor='left', yanchor='top', pad=dict(t=30),
                      buttons=[dict(label='▶ Percorrer os 32 filtros', method='animate',
                                    args=[None, dict(frame=dict(duration=900, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                               dict(label='⏸ Pausar', method='animate',
                                    args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

print('Filtro 1 (linhas x colunas), os mesmos 9 pesos das barras iniciais:')
print(np.round(K[0], 3))

tab = pd.DataFrame({'Filtro': np.arange(1, nf + 1),
                    'Menor peso': [m.min() for m in K], 'Maior peso': [m.max() for m in K],
                    'Soma dos 9 pesos': [m.sum() for m in K],
                    'Maior |peso|': [np.abs(m).max() for m in K]}).set_index('Filtro')
print('\nOs 8 filtros com maior |peso|:')
display(tab.sort_values('Maior |peso|', ascending=False).head(8).round(3))

Filtro 1 (linhas x colunas), os mesmos 9 pesos das barras iniciais:
[[-0.587 -1.216 -0.426]
 [-0.079 -1.236 -0.383]
 [-1.146 -1.341 -0.125]]

Os 8 filtros com maior |peso|:


,Menor peso,Maior peso,Soma dos 9 pesos,Maior |peso|
Filtro,,,,
1,-1.341,-0.079,-6.539,1.341
14,-0.694,1.112,-0.034,1.112
4,-0.101,1.082,4.519,1.082
31,-0.903,1.034,-0.045,1.034
18,-0.356,1.032,3.746,1.032
28,-0.975,1.028,2.273,1.028
10,-1.013,0.898,-0.036,1.013
12,-0.857,1.005,-0.034,1.005


**Observação:** o gráfico mostra que cada um dos 32 filtros da Conv 1 é uma grade minúscula de apenas nove pesos (32 × 9 = 288 pesos, como no inventário do Passo 5), e que esses nove números são tudo que a rede usa, em cada filtro, para olhar uma vizinhança de 3×3 pixels da imagem. A escala de cores é comum a todos os filtros e vai de cerca de −1,4 a +1,4, de acordo com a faixa de pesos que vimos no Passo 7 (de −1,34 a +1,11), o que permite comparar a intensidade de um filtro com a de outro apenas pela cor e pela altura das barras. O filtro que aparece na tela tem barras predominantemente azuis, isto é, pesos negativos ou próximos de zero, com uma barra bem mais profunda que as demais; ao passar de um filtro para outro, o padrão de altura e de cor muda, o que mostra que os filtros não são cópias uns dos outros. Há uma limitação importante: nove barras mostram os valores dos pesos, mas não dizem o que o filtro detecta. Para saber se um filtro responde a bordas, bordas escuras ou regiões claras, é preciso aplicá-lo a uma imagem e olhar o mapa de ativação, o que faremos mais adiante no Gráfico 15.

## Fase 3 — Ativações: o que a rede faz com o que calcula

Até aqui olhamos para os parâmetros da rede, que são os números guardados depois do treino. A partir desta fase olhamos para o que a rede produz quando recebe uma imagem. O primeiro elemento é a função de ativação: depois de cada soma ponderada, ela decide o que passa para a camada seguinte.

### Passo 12 — Gráfico 13: funções de ativação e suas derivadas

O Gráfico 13 do catálogo compara funções de ativação. Aqui ele aparece em dois painéis lado a lado: à esquerda, a função f(z), isto é, a saída em função da soma ponderada z; à direita, a derivada f'(z), isto é, a inclinação da curva, que é o que o aprendizado usa para decidir quanto ajustar cada peso. São seis funções: ReLU, Leaky ReLU, ELU, GELU, sigmoide e tanh. A ReLU, em dourado e mais grossa, é a que o nosso modelo usa depois de cada uma das quatro convoluções. A camada de saída usa a softmax, que não aparece aqui porque não é uma curva de um número só: ela transforma as quatro saídas em probabilidades que somam 1. O gráfico é animado: a barra deslizante, ou o botão "Mover z", faz um ponto percorrer cada curva, para você ler a saída e a inclinação de cada função para o mesmo z. Antes do gráfico, o código confirma no próprio modelo quantas camadas ReLU ele tem e qual é a ativação da saída.

In [14]:
# Passo 12 — Gráfico 13: funções de ativação e suas derivadas
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import erf

# o que o modelo realmente usa
n_relu = sum(isinstance(l, keras.layers.ReLU) for l in modelo.layers)
print(f'Camadas ReLU no modelo: {n_relu}')
print(f'Ativação da camada de saída: {modelo.layers[-1].activation.__name__}')

x = np.linspace(-5, 5, 401)
sig = lambda z: 1 / (1 + np.exp(-z))
fun = {
    'ReLU (usada no modelo)': (lambda z: np.maximum(0, z),            lambda z: (z > 0) * 1.0),
    'Leaky ReLU':             (lambda z: np.where(z > 0, z, 0.1 * z), lambda z: np.where(z > 0, 1.0, 0.1)),
    'ELU':                    (lambda z: np.where(z > 0, z, np.exp(z) - 1), lambda z: np.where(z > 0, 1.0, np.exp(z))),
    'GELU':                   (lambda z: 0.5 * z * (1 + erf(z / np.sqrt(2))),
                               lambda z: 0.5 * (1 + erf(z / np.sqrt(2))) + z * np.exp(-z**2 / 2) / np.sqrt(2 * np.pi)),
    'Sigmoide':               (sig,        lambda z: sig(z) * (1 - sig(z))),
    'Tanh':                   (np.tanh,    lambda z: 1 - np.tanh(z) ** 2),
}
cores = ['#C9A227', '#1F4E79', '#2E8B57', '#8E44AD', '#C0392B', '#7F8C8D']

fig = make_subplots(rows=1, cols=2, subplot_titles=('Função  f(z)', 'Derivada  f\'(z)'), horizontal_spacing=0.08)
for (nome, (f, d)), cor in zip(fun.items(), cores):
    larg = 9 if nome.startswith('ReLU') else 2.5
    fig.add_trace(go.Scatter(x=x, y=f(x), name=nome, mode='lines', line=dict(color=cor, width=larg), legendgroup=nome), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=d(x), name=nome, mode='lines', line=dict(color=cor, width=larg), legendgroup=nome, showlegend=False), row=1, col=2)

# marcadores móveis (um por função, em cada painel)
z0 = -5.0
for k, ((nome, (f, d)), cor) in enumerate(zip(fun.items(), cores)):
    fig.add_trace(go.Scatter(x=[z0], y=[f(np.array([z0]))[0]], mode='markers', showlegend=False, legendgroup=nome,
                             marker=dict(color=cor, size=11, line=dict(color='white', width=2)),
                             hovertemplate=nome + '<br>f(z) = %{y:.3f}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[z0], y=[d(np.array([z0]))[0]], mode='markers', showlegend=False, legendgroup=nome,
                             marker=dict(color=cor, size=11, line=dict(color='white', width=2)),
                             hovertemplate=nome + '<br>f\'(z) = %{y:.3f}<extra></extra>'), row=1, col=2)

indices = list(range(12, 24))
valores = np.round(np.arange(-5, 5.01, 0.25), 2)
quadros, passos = [], []
for v in valores:
    dados = []
    for nome, (f, d) in fun.items():
        dados += [go.Scatter(x=[v], y=[f(np.array([v]))[0]]), go.Scatter(x=[v], y=[d(np.array([v]))[0]])]
    quadros.append(go.Frame(name=str(v), data=dados, traces=indices))
    passos.append(dict(method='animate', label=f'{v:g}',
                       args=[[str(v)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = quadros

fig.update_xaxes(title_text='Entrada z (soma ponderada)', range=[-5, 5], zeroline=True)
fig.update_yaxes(range=[-1.5, 5], row=1, col=1, title_text='Saída')
fig.update_yaxes(range=[-0.3, 1.3], row=1, col=2, title_text='Inclinação')
fig.update_layout(title='Gráfico 13 — Funções de ativação e suas derivadas (a destacada é a usada nas convoluções do modelo)',
                  template='plotly_white', height=720, margin=dict(b=200), legend=dict(orientation='h', y=-0.62),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='z = '), pad=dict(t=30), y=-0.12)],
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=-0.12, xanchor='left', yanchor='top', pad=dict(t=0, r=10),
                                    buttons=[dict(label='▶ Mover z', method='animate',
                                                  args=[None, dict(frame=dict(duration=250, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

Camadas ReLU no modelo: 4
Ativação da camada de saída: softmax


**Observação:** o gráfico mostra duas famílias de comportamento. A ReLU, que é a função usada depois de cada convolução do modelo, vale zero para qualquer soma ponderada negativa e é igual à própria soma quando ela é positiva; por isso sua derivada é 0 do lado esquerdo e 1 do lado direito, com um salto em z = 0. Em termos práticos, quando z é negativo o neurônio fica "desligado" e não passa nada adiante (nem recebe ajuste no aprendizado), e quando z é positivo ele repassa o valor sem alterá-lo. A Leaky ReLU deixa passar uma fração pequena do lado negativo, com inclinação constante de 0,1, e a ELU e a GELU suavizam a transição perto de zero; a GELU chega a ter derivada ligeiramente negativa por volta de z = −1,5 e ligeiramente acima de 1 depois de zero. As outras duas, sigmoide e tanh, são curvas que saturam: a sigmoide fica entre 0 e 1 e a tanh entre −1 e 1, e nas duas a derivada tende a zero nas pontas (z muito negativo ou muito positivo), com pico de 0,25 na sigmoide e de 1 na tanh, ambos em z = 0. Em redes com muitas camadas, derivadas muito pequenas multiplicadas camada após camada dificultam o aprendizado, e é uma das razões da popularidade da ReLU, cuja derivada vale 1 em toda a metade positiva. Esta leitura compara as formas das curvas e não mede, por si só, qual função é melhor para o nosso problema.

### Passo 13 — Gráfico 14: a superfície de ativação de um neurônio

O Gráfico 14 do catálogo mostra, em três dimensões, o que um neurônio faz com suas entradas. Aqui ele é feito com os filtros reais da primeira camada. Cada filtro olha para nove pixels ao mesmo tempo; para poder desenhar em 3D, escolhemos os dois pixels (A e B) com maior peso em valor absoluto em cada filtro, deixamos variar de 0 (preto) a 1 (branco), que é a escala da entrada depois do Rescaling, e fixamos os outros sete pixels em 0. Com isso, a soma ponderada do neurônio é z = peso A · A + peso B · B + bias, com os valores reais do modelo. No gráfico há duas superfícies: a cinza e transparente é z, antes da ativação, que é sempre um plano; a colorida é a saída depois da ReLU, o plano "dobrado" que corta tudo que fica abaixo de zero. Onde o plano cinza fica acima de zero as duas coincidem; onde ele fica abaixo, a superfície colorida fica achatada em zero, e é aí que o neurônio está "desligado". A barra deslizante e o botão "Percorrer os 32 filtros" trocam de filtro, o título mostra a fórmula do filtro mostrado, e você pode girar o gráfico com o mouse. A tabela impressa abaixo traz, para cada filtro, os dois pixels usados, os pesos, o bias e a porcentagem da grade de entradas em que o neurônio fica ligado. É um recorte de duas entradas dentro de nove, e serve para entender a mecânica da ativação, não para resumir o filtro inteiro.

In [15]:
# Passo 13 — Gráfico 14: superfície de ativação de um neurônio da Conv 1 (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

conv1 = [l for l in modelo.layers if l.weights][0]
W1, b1 = conv1.get_weights()                       # W1: (3, 3, 1, 32)   b1: (32,)
nf = W1.shape[3]

g = np.linspace(0, 1, 41)                          # entrada normalizada: 0 = preto, 1 = branco
X1, X2 = np.meshgrid(g, g)

dados, linhas = [], []
for f in range(nf):
    w = W1[:, :, 0, f].ravel()
    a, b = np.argsort(-np.abs(w))[:2]              # os dois pixels com maior |peso| do filtro
    z = w[a] * X1 + w[b] * X2 + b1[f]              # soma ponderada (os outros 7 pixels fixos em 0)
    dados.append((z, np.maximum(0, z), a, b, w[a], w[b], b1[f]))
    linhas.append({'Filtro': f + 1, 'Pixel A (lin,col)': f'({a//3+1},{a%3+1})', 'Peso A': w[a],
                   'Pixel B (lin,col)': f'({b//3+1},{b%3+1})', 'Peso B': w[b], 'Bias': b1[f],
                   '% da grade com neurônio ligado': 100 * np.mean(z > 0)})

zmin = min(d[0].min() for d in dados); zmax = max(d[0].max() for d in dados)

def titulo(f):
    z, r, a, b, wa, wb, bias = dados[f]
    return f'Gráfico 14 — Filtro {f+1}: z = ({wa:.2f})·A + ({wb:.2f})·B + ({bias:.2f}),  saída = ReLU(z)'

plano = go.Surface(x=g, y=g, z=dados[0][0], opacity=0.35, showscale=False, colorscale=[[0, '#9AA5B1'], [1, '#9AA5B1']],
                   name='z (antes da ReLU)', hovertemplate='A=%{x:.2f}  B=%{y:.2f}<br>z=%{z:.3f}<extra>antes da ReLU</extra>')
relu = go.Surface(x=g, y=g, z=dados[0][1], colorscale=[[0, '#1F4E79'], [1, '#C9A227']], cmin=0, cmax=max(zmax, 0.01),
                  colorbar=dict(title='ReLU(z)'), name='ReLU(z)',
                  hovertemplate='A=%{x:.2f}  B=%{y:.2f}<br>saída=%{z:.3f}<extra>depois da ReLU</extra>')

quadros = [go.Frame(name=str(f + 1), traces=[0, 1], data=[go.Surface(z=dados[f][0]), go.Surface(z=dados[f][1])],
                    layout=go.Layout(title_text=titulo(f))) for f in range(nf)]
passos = [dict(method='animate', label=str(f + 1),
               args=[[str(f + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
          for f in range(nf)]

fig = go.Figure(data=[plano, relu], frames=quadros)
fig.update_layout(title=titulo(0), template='plotly_white', height=720,
                  scene=dict(xaxis_title='Pixel A', yaxis_title='Pixel B',
                             zaxis=dict(title='z e saída', range=[min(zmin, 0), max(zmax, 0.01)]),
                             camera=dict(eye=dict(x=1.7, y=-1.7, z=0.9))),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Filtro '), pad=dict(t=90))],
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=0.0, xanchor='left', yanchor='top', pad=dict(t=30),
                                    buttons=[dict(label='▶ Percorrer os 32 filtros', method='animate',
                                                  args=[None, dict(frame=dict(duration=900, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

tab = pd.DataFrame(linhas).set_index('Filtro')
display(tab.round(3))

,"Pixel A (lin,col)",Peso A,"Pixel B (lin,col)",Peso B,Bias,% da grade com neurônio ligado
Filtro,,,,,,
1,"(3,2)",-1.341,"(2,2)",-1.236,0.069,0.357
2,"(3,1)",-0.848,"(2,1)",-0.587,-0.012,0.000
3,"(1,2)",-0.698,"(2,1)",0.630,-0.046,39.024
4,"(2,1)",1.082,"(2,2)",0.899,0.001,100.000
5,"(3,2)",0.893,"(2,2)",-0.636,-0.009,63.058
6,"(3,1)",-0.851,"(1,1)",0.619,-0.055,30.637
7,"(3,1)",-0.926,"(3,3)",0.827,-0.021,42.594
8,"(2,2)",-0.663,"(1,1)",0.609,-0.008,44.854
9,"(3,1)",0.932,"(2,2)",-0.522,0.016,73.111


**Observação:** a superfície mostra o que a ReLU faz com a soma ponderada de dois pixels. No filtro 1, os dois pesos são negativos (−1,341 e −1,236) e o bias é pequeno e positivo (0,069); por isso z só fica acima de zero quando os dois pixels estão praticamente pretos, e o neurônio fica ligado em apenas 0,357% da grade de entradas, com a superfície colorida achatada em zero em quase toda a região (por exemplo, com A = 0,35 e B = 0,05 o valor de z é −0,462 e a saída é zero). O filtro 2 é parecido, com os dois pesos negativos (−0,848 e −0,587) e 0,000% de grade ligada. O comportamento oposto aparece nos filtros 4 e 13, que têm os dois pesos positivos (1,082 e 0,899; 0,660 e 0,645) e ficam ligados em 100,000% e 99,822% da grade: nesses casos a ReLU quase não corta nada e o neurônio repassa o brilho dos pixels. A maior parte dos filtros que aparecem na tabela (do 3 ao 12, o 14 e o 15) tem um peso positivo e outro negativo, e fica ligada em frações intermediárias da grade, de 30,637% (filtro 6) a 73,111% (filtro 9). Esses neurônios respondem à diferença entre os dois pixels e não ao brilho de cada um: crescem quando o pixel de peso positivo é mais claro que o de peso negativo, e a ReLU corta o resto. Os biases vistos são pequenos (de −0,055 a +0,069) e, por isso, quem decide o formato da superfície são os pesos, e o bias só desloca o ponto de corte. Duas ressalvas: a porcentagem de grade ligada depende da escolha de fixar os outros sete pixels em 0 e de uma grade uniforme de 0 a 1, e imagens reais não têm pixels distribuídos assim; e o gráfico mostra dois dos nove pixels de cada filtro, então não resume o comportamento completo dele. O que o Gráfico 15 vai mostrar, com imagens reais, é o que acontece quando os nove pixels variam juntos.

### Passo 14 — As imagens de exemplo

Os próximos gráficos da fase (15, 16 e 17) mostram o que acontece dentro da rede quando ela recebe um exame de verdade. Para isso precisamos de exemplos fixos. Aqui escolhemos cinco imagens do conjunto de validação, nunca do teste: uma de cada uma das quatro classes (glioma, meningioma, sem tumor e pituitária), sorteadas com semente fixa entre as que têm formato original quadrado, e uma quinta imagem de "sem tumor" com formato original retangular. Essa quinta serve para ilustrar o atalho que apareceu no projeto original: imagens retangulares ganham bandas pretas ao serem preparadas para 128×128, e o formato acabou associado à classe "sem tumor". O sorteio é repetível, então os mesmos exemplos aparecem toda vez que o notebook for executado. O código mostra as cinco imagens como a rede as vê (já em escala de cinza e com 128×128 pixels), imprime uma tabela com o tamanho original do arquivo, a previsão do modelo e a probabilidade que ele deu a cada classe, e guarda as imagens escolhidas para os próximos passos. Um aviso: o modelo acerta cerca de 91% da validação, então é possível que algum exemplo seja classificado errado, e isso também será informativo.

In [16]:
# Passo 14 — As 5 imagens de exemplo (validação) e a previsão do modelo
import numpy as np
import pandas as pd
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# tamanho original de cada imagem de validação (só o cabeçalho é lido, não os pixels)
tam = [Image.open(c).size for c in val['caminho']]
val_t = val.copy()
val_t['largura'] = [t[0] for t in tam]
val_t['altura'] = [t[1] for t in tam]
val_t['quadrada'] = val_t['largura'] == val_t['altura']

# 1 imagem por classe (originais quadradas), sorteio fixo e repetível
rng = np.random.default_rng(42)
idx_ex, nomes = [], []
for cls in CLASSES:
    cand = val_t[(val_t['classe'] == cls) & val_t['quadrada']]
    if len(cand) == 0:
        cand = val_t[val_t['classe'] == cls]
    idx_ex.append(int(rng.choice(cand.index.values)))
    nomes.append(f'Exemplo — {cls}')

# 5ª imagem: "sem tumor" retangular (com bandas pretas ao preparar), para ilustrar o atalho do projeto original
cand = val_t[(val_t['classe'] == 'notumor') & (~val_t['quadrada'])]
if len(cand) > 0:
    idx_ex.append(int(rng.choice(cand.index.values)))
    nomes.append('Exemplo — notumor retangular')
else:
    print('Não há imagem retangular de notumor na validação; ficam só as 4 imagens.')

X_ex = X_val[idx_ex].astype('float32')                    # imagens já preparadas (128x128x1), as mesmas que a rede vê
prob = modelo.predict(X_ex, verbose=0)
pred = prob.argmax(axis=1)

linhas = []
for k, i in enumerate(idx_ex):
    r = val_t.loc[i]
    linhas.append({'Exemplo': nomes[k], 'Classe real': r['classe'], 'Arquivo': r['arquivo'],
                   'Original (larg×alt)': f"{r['largura']}×{r['altura']}",
                   'Previsão do modelo': CLASSES[pred[k]], 'Acertou': 'sim' if CLASSES[pred[k]] == r['classe'] else 'NÃO',
                   **{f'P({c})': prob[k, j] for j, c in enumerate(CLASSES)}})
tab = pd.DataFrame(linhas).set_index('Exemplo')

titulos = [f"{nomes[k].replace('Exemplo — ', '')}<br>prev.: {CLASSES[pred[k]]} ({prob[k, pred[k]]:.0%})" for k in range(len(idx_ex))]
fig = make_subplots(rows=1, cols=len(idx_ex), subplot_titles=titulos, horizontal_spacing=0.02)
for k, i in enumerate(idx_ex):
    fig.add_trace(go.Heatmap(z=X_val[i, :, :, 0], colorscale='gray', showscale=False,
                             hovertemplate='linha %{y}, coluna %{x}<br>pixel %{z}<extra></extra>'), row=1, col=k + 1)
    fig.update_xaxes(showticklabels=False, row=1, col=k + 1)
    fig.update_yaxes(autorange='reversed', showticklabels=False, scaleanchor=f'x{k+1}' if k else 'x', row=1, col=k + 1)
fig.update_layout(title='Passo 14 — As imagens de exemplo (validação), como a rede as vê (128×128)',
                  template='plotly_white', height=360, margin=dict(t=110))
fig.show()

display(tab.round(3))

,Classe real,Arquivo,Original (larg×alt),Previsão do modelo,Acertou,P(glioma),P(meningioma),P(notumor),P(pituitary)
Exemplo,,,,,,,,,
Exemplo — glioma,glioma,Tr-gl_111.jpg,512×512,glioma,sim,0.999,0.000,0.001,0.000
Exemplo — meningioma,meningioma,Tr-me_711.jpg,512×512,glioma,NÃO,0.511,0.478,0.012,0.000
Exemplo — notumor,notumor,Tr-no_552.jpg,630×630,notumor,sim,0.000,0.000,1.000,0.000
Exemplo — pituitary,pituitary,Tr-pi_29.jpg,512×512,pituitary,sim,0.000,0.000,0.000,1.000
Exemplo — notumor retangular,notumor,Tr-no_258.jpg,1375×1446,notumor,sim,0.000,0.072,0.921,0.007
